# Deconstructing the GPT Model Architecture

In the last notebook, we transformed a raw corpus of text into batches of numerical tokens. Now, we're ready to build the machine that will learn from this data: the GPT model itself. This is the "brain" of our operation, and understanding its internal structure is key to understanding how language models work.

By the end of this notebook, you will be able to implement a Causal Self-Attention mechanism from scratch, understand the roles of Layer Normalization and the MLP, and assemble these components into a complete Transformer Block—the fundamental building block of a GPT model.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

# A simplified config for our model
class GPTConfig:
    vocab_size: int = 50257 # Vocabulary size from GPT-2
    block_size: int = 128   # Max sequence length
    n_embd: int = 256       # Embedding dimension
    n_head: int = 4         # Number of attention heads
    n_layer: int = 4        # Number of transformer blocks
    dropout: float = 0.1

config = GPTConfig()

### The Core Idea: A Committee of Specialists

Imagine you're trying to predict the next word in the sentence: "The cat sat on the ___". To make a good guess, you need to understand the context. The word "sat" is probably the most important clue, but "cat" is also relevant.

A Transformer block works like a committee meeting where each token in the sentence is a member. The goal of the meeting is for each member to enrich their understanding of their own role in the sentence by communicating with the others. This meeting happens in two main phases.

1.  **Phase 1: Communication (Self-Attention)**. Each member (token) looks at every other member that came before it. It asks three questions:
    *   **Query**: "Here is what I'm looking for."
    *   **Key**: "Here is what I have to offer."
    *   **Value**: "Here is the information I will share if you're interested."

    A token pays more "attention" to other tokens where its *Query* matches their *Key*. It then takes a weighted average of their *Values*. This allows the token for "sat" to strongly connect with the token for "cat", gathering crucial context. In GPT, this is "causal" attention, meaning tokens can only look at those that appeared earlier in the sentence, ensuring we don't cheat when predicting the next word.

2.  **Phase 2: Individual Thinking (MLP)**. After the group discussion, each member (token) goes back to its own "office" to think. This is the Multi-Layer Perceptron (MLP). It takes the rich, context-aware information from the attention phase and processes it further. Each token is processed independently. It's like each committee member digesting the meeting's discussion to form a more sophisticated final thought.

Sprinkled throughout this process are two other crucial elements:
-   **Residual Connections**: Imagine taking notes. Before the meeting starts, you have your initial thoughts. After the discussion, you *add* the new insights to your original notes instead of replacing them. This is a residual connection (`output = input + new_info`). It helps the model learn by making it easier for information and gradients to flow through deep networks.
-   **Layer Normalization**: This is like a moderator in the meeting who ensures everyone is speaking at roughly the same volume. It stabilizes the numbers flowing through the model, making training much more reliable.

A single Transformer "Block" is just one round of this meeting. A full GPT model is just a stack of these blocks, one after another. Each subsequent meeting allows the tokens to build ever more complex and nuanced representations of the text.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        
        # Key, Query, Value projections
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        
        # Causal mask
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # Batch size, Sequence length, Embedding dimensionality

        # Get query, key, value for all heads in batch
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        # Reshape for multi-head attention: (B, T, C) -> (B, n_head, T, head_size)
        head_size = C // self.n_head
        k = k.view(B, T, self.n_head, head_size).transpose(1, 2)
        q = q.view(B, T, self.n_head, head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_size).transpose(1, 2)

        # Attention mechanism
        # (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        
        # Concatenate heads and project back
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        # The two core parts of the block, with residual connections
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

### Walking Through the Implementation

Let's break down the `Block` class, which is the heart of the model. Its `forward` method is surprisingly simple and clearly shows the architecture we described.

`x = x + self.attn(self.ln_1(x))`

1.  `self.ln_1(x)`: First, we apply **Layer Normalization** to the input `x`. This stabilizes the activations, acting like the "moderator" ensuring a clean start to the communication phase.
2.  `self.attn(...)`: The normalized output is then fed into the **Causal Self-Attention** module. This is the "committee discussion" where tokens gather context from their predecessors.
3.  `x + ...`: The output from the attention mechanism is then **added** back to the original, unmodified input `x`. This is the crucial **residual connection** (or "skip connection"). It ensures that we are *adding new information* rather than completely transforming the original signal, which helps with training stability and performance.

`x = x + self.mlp(self.ln_2(x))`

This line follows the exact same pattern, but for the "individual thinking" phase.

1.  `self.ln_2(x)`: We take the output of the first stage and normalize it again.
2.  `self.mlp(...)`: The normalized tensor is passed to the **Multi-Layer Perceptron**. Each token's representation is processed independently by this small neural network.
3.  `x + ...`: And again, the output of the MLP is added back to its input, forming the second residual connection.

The result is a tensor of the exact same shape as the input, but with its token representations now enriched with contextual information from this "meeting". A full GPT model simply stacks these `Block` modules one on top of another.

In [ ]:
# Demonstrate a single block in action

# Create some dummy data
batch_size = 4
sequence_length = config.block_size
embedding_dim = config.n_embd

# A random tensor representing a batch of token embeddings
dummy_input = torch.randn(batch_size, sequence_length, embedding_dim)

# Instantiate a single Transformer Block
transformer_block = Block(config)

# Pass the dummy data through the block
output = transformer_block(dummy_input)

print("Shape of the input:", dummy_input.shape)
print("Shape of the output:", output.shape)
print("\nNotice the shape is preserved. The block enriches the embeddings, it doesn't change their dimensions.")

### Going Deeper: Assembling the Full GPT Model

A single `Block` is powerful, but a real GPT model needs more. It needs a way to convert token IDs into vectors, a way to understand token positions, and a way to turn the final enriched vectors back into word predictions. Let's assemble our `Block` into a minimal, but complete, GPT model.

The full model performs these steps:
1.  **Token and Positional Embeddings**: It looks up the vector for each token ID in the input (`wte`) and adds a corresponding vector for its position in the sequence (`wpe`). The position vector is crucial; without it, the model would be a "bag of words" and wouldn't know the order of the text.
2.  **Transformer Blocks**: The resulting combined embeddings are passed through a series of `Block` modules (`h`), just like the one we built. Each block deepens the model's understanding.
3.  **Final Normalization**: A final `LayerNorm` (`ln_f`) is applied after the last block.
4.  **Language Model Head**: A final linear layer (`lm_head`) takes the enriched token vectors and projects them up to the size of our entire vocabulary. The output values, called "logits," represent the model's prediction for the next token at each position.

This structure shows how the simple `Block` we designed is the core repeating unit within a much larger and more capable system.

In [ ]:
class MinimalGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        # The model is a dictionary of modules
        self.transformer = nn.ModuleDict(dict(
            # Token Embedding Lookup Table
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            # Position Embedding Lookup Table
            wpe = nn.Embedding(config.block_size, config.n_embd),
            # A stack of N transformer blocks
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            # Final LayerNorm
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        # The final layer to project back to vocabulary size
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        
        # Tie the weights of the token embedding and the final layer
        # This is a common practice that improves performance
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx):
        B, T = idx.size()
        assert T <= self.config.block_size, f"Sequence length {T} exceeds block size {self.config.block_size}"
        
        # 1. Get token and position embeddings
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device).unsqueeze(0) # shape (1, T)
        tok_emb = self.transformer.wte(idx) # (B, T, n_embd)
        pos_emb = self.transformer.wpe(pos) # (1, T, n_embd)
        x = tok_emb + pos_emb

        # 2. Pass through the stack of transformer blocks
        for block in self.transformer.h:
            x = block(x)

        # 3. Final normalization and language model head
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)

        return logits

# Instantiate the full model
model = MinimalGPT(config)
print(f"Model created with {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")

# Create a dummy input of token IDs
dummy_indices = torch.randint(0, config.vocab_size, (batch_size, sequence_length))

# Get the model's predictions (logits)
logits = model(dummy_indices)
print("Shape of model input (token IDs):", dummy_indices.shape)
print("Shape of model output (logits):", logits.shape)

### Visualization: The Data Flow in a Transformer Block

Words can be tricky, so let's visualize the data path inside a single `Block`. The most important concepts to see are the **two main branches** (Attention and MLP) and the **residual connections** that bypass them.

The diagram below shows how the input tensor `x` flows through the block. Notice how at two points, a version of `x` is carried forward on a "shortcut" path and added back in. This ensures that the block's job is always to *refine* the representation, not to create it from scratch. This simple architectural trick is one of the main reasons deep transformers can be trained effectively.

In [ ]:
def plot_transformer_block():
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')

    # Styles
    box_style = dict(boxstyle="round,pad=0.5", fc="lightblue", ec="b", lw=2)
    arrow_style = dict(arrowstyle="->", connectionstyle="arc3,rad=0.1", ec="black", lw=2)
    
    # Nodes
    nodes = {
        'in': (0, 5), 'ln1': (0, 4), 'attn': (0, 3), 'add1': (0, 2),
        'ln2': (0, 1), 'mlp': (0, 0), 'add2': (0, -1), 'out': (0, -2)
    }
    
    # Draw nodes
    ax.text(nodes['in'][0], nodes['in'][1], "Input (x)", ha="center", va="center", bbox=box_style, size=12)
    ax.text(nodes['ln1'][0], nodes['ln1'][1], "LayerNorm", ha="center", va="center", bbox=box_style, size=12)
    ax.text(nodes['attn'][0], nodes['attn'][1], "Causal Self-Attention", ha="center", va="center", bbox=box_style, size=12)
    ax.text(nodes['add1'][0], nodes['add1'][1], "+", ha="center", va="center", bbox=dict(boxstyle="circle", fc="lightgreen"), size=18)
    ax.text(nodes['ln2'][0], nodes['ln2'][1], "LayerNorm", ha="center", va="center", bbox=box_style, size=12)
    ax.text(nodes['mlp'][0], nodes['mlp'][1], "MLP", ha="center", va="center", bbox=box_style, size=12)
    ax.text(nodes['add2'][0], nodes['add2'][1], "+", ha="center", va="center", bbox=dict(boxstyle="circle", fc="lightgreen"), size=18)
    ax.text(nodes['out'][0], nodes['out'][1], "Output", ha="center", va="center", bbox=box_style, size=12)
    
    # Main path arrows
    ax.annotate("", xy=nodes['ln1'], xytext=nodes['in'], arrowprops=arrow_style)
    ax.annotate("", xy=nodes['attn'], xytext=nodes['ln1'], arrowprops=arrow_style)
    ax.annotate("", xy=nodes['add1'], xytext=nodes['attn'], arrowprops=arrow_style)
    ax.annotate("", xy=nodes['ln2'], xytext=nodes['add1'], arrowprops=arrow_style)
    ax.annotate("", xy=nodes['mlp'], xytext=nodes['ln2'], arrowprops=arrow_style)
    ax.annotate("", xy=nodes['add2'], xytext=nodes['mlp'], arrowprops=arrow_style)
    ax.annotate("", xy=nodes['out'], xytext=nodes['add2'], arrowprops=arrow_style)
    
    # Residual connections
    ax.annotate("", xy=nodes['add1'], xytext=(nodes['in'][0]-1, nodes['in'][1]), 
                arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=-0.5", ec="red", lw=2, ls="--"))
    ax.annotate("", xy=nodes['add2'], xytext=(nodes['add1'][0]-1, nodes['add1'][1]), 
                arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=-0.5", ec="red", lw=2, ls="--"))
    
    ax.text(-1.5, 3.5, "Residual Connection", ha="center", va="center", rotation=90, size=10, color="red")
    ax.text(-1.5, -0.5, "Residual Connection", ha="center", va="center", rotation=90, size=10, color="red")

    plt.title("Data Flow Inside a Transformer Block", size=16)
    plt.ylim(-2.5, 5.5)
    plt.xlim(-2, 2)
    plt.show()

plot_transformer_block()

### Summary & What's Next

In this notebook, we moved from abstract concepts to concrete code, building the architectural heart of a GPT model.

**What We Built**:
- A `CausalSelfAttention` module that allows tokens to communicate with their past.
- An `MLP` module to allow for independent computation on each token.
- A `Block` that combines attention, MLP, layer normalization, and residual connections into a single, powerful, and reusable unit.
- A `MinimalGPT` model that stacks these blocks and adds the necessary embedding and output layers to form a complete language model.

**Key Takeaways**:
- **Composition is key**: A complex GPT model is built by repeatedly stacking a simple, well-designed `Block`.
- **Attention is for communication**: The self-attention mechanism is how the model builds context, figuring out which words in the sequence are important to which others.
- **Residual connections are vital**: These "shortcuts" are not a minor detail; they are essential for enabling the training of deep, multi-layer transformer models.

We now have a complete model architecture. It's initialized with random weights, so it doesn't know anything yet. The next step is to teach it. In the upcoming notebook, **"Orchestrating the LLM Training Loop,"** we will write the code that feeds our model the data we prepared, evaluates its predictions, and updates its weights to make it smarter.